# 3.1 Boto3 — AWS SDK for Python> ☕ **Java parallel:** Like the AWS SDK for Java, but more concise.---

## S3 Operations — Data Engineering Essentials

In [ ]:
import boto3from botocore.exceptions import ClientError# NOTE: Requires AWS credentials configured# aws configure  (or env vars AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)# S3 clients3 = boto3.client('s3')# --- Common S3 operations for data engineering ---def upload_data(bucket: str, key: str, data: str) -> bool:    """Upload data to S3."""    try:        s3.put_object(Bucket=bucket, Key=key, Body=data.encode())        return True    except ClientError as e:        print(f"Upload failed: {e}")        return Falsedef list_partitions(bucket: str, prefix: str) -> list[str]:    """List partition prefixes — common for Hive-style partitioned data."""    paginator = s3.get_paginator('list_objects_v2')    partitions = set()    for page in paginator.paginate(Bucket=bucket, Prefix=prefix, Delimiter='/'):        for cp in page.get('CommonPrefixes', []):            partitions.add(cp['Prefix'])    return sorted(partitions)def read_parquet_from_s3(bucket: str, key: str):    """Read a Parquet file from S3 into Pandas."""    import pandas as pd    import io    obj = s3.get_object(Bucket=bucket, Key=key)    return pd.read_parquet(io.BytesIO(obj['Body'].read()))print("Boto3 S3 helpers defined — run with real AWS credentials")

## EMR — Submitting Spark Jobs☕ You know EMR from your Oracle work. Here's how to do it programmatically.

In [ ]:
# EMR job submission — programmatic version of what you did at Oracleemr = boto3.client('emr', region_name='us-east-1')def submit_spark_job(cluster_id: str, script_path: str, args: list[str] = None):    """Submit a Spark job to an existing EMR cluster."""    step = {        'Name': 'Healthcare ETL',        'ActionOnFailure': 'CONTINUE',        'HadoopJarStep': {            'Jar': 'command-runner.jar',            'Args': [                'spark-submit',                '--deploy-mode', 'cluster',                '--master', 'yarn',                script_path,                *(args or []),            ]        }    }    # response = emr.add_job_flow_steps(JobFlowId=cluster_id, Steps=[step])    print(f"Would submit: {step['Name']} to cluster {cluster_id}")    return stepsubmit_spark_job('j-XXXXX', 's3://bucket/scripts/transform.py', ['--date', '2024-01-15'])

## Glue — Serverless ETLAWS Glue uses PySpark under the hood — your Spark knowledge applies directly.

In [ ]:
# Glue Data Catalog — query tables and partitionsglue = boto3.client('glue', region_name='us-east-1')def get_table_schema(database: str, table: str) -> dict:    """Fetch table schema from Glue Data Catalog."""    try:        response = glue.get_table(DatabaseName=database, Name=table)        columns = response['Table']['StorageDescriptor']['Columns']        return {col['Name']: col['Type'] for col in columns}    except ClientError as e:        print(f"Error: {e}")        return {}print("Glue helpers defined")# 💡 Tip: Use awswrangler (pip install awswrangler) for simpler# Pandas <-> S3/Glue/Athena integration# import awswrangler as wr# df = wr.s3.read_parquet('s3://bucket/path/')# wr.s3.to_parquet(df, 's3://bucket/output/')